# StealthDiffusion · M2 第一个小验证
**Theme 1 / Dark.ModelBreakers · 10 张图 · 解码前后配对对照**

本版由你上传的 `StealthDiffusion_Baseline_Submission_1.ipynb` 第 3 节官方运行流程扩展：
沿用 Python 3.10 环境、六项权重以及官方 LAO / Control-VAE，增加最终潜变量的观察与独立评测。
原 M1 文件保持不变；本版只包含这次小验证的可读 Python 源码。
从上到下运行：填写路径 → 保存下面可读的 Python 模块 → 配置环境 → 运行与查看结果。

**当前状态：诊断工具已编写；尚未在真实 A100 / 图像 / 权重上执行。本 Notebook 没有预填结果。**

研究问题：同一个最终优化潜变量，在普通 VAE 上能逃逸的图像，经过最终 Control-VAE 路径后是否重新被识别？
若出现这一现象，再尝试把 LAO 损失移到最终解码路径，进行原版 / 改进版比较。
这次先验证该改动的动机，不训练新模型，也不加入 CLIP 或 EOT。


## 1. 本次对照

| 输出 | 含义 |
|---|---|
| Clean | 同一张图经官方 Lanczos 缩放后的 224×224 输入 |
| Plain VAE | 最后一次 latent 更新之后，按原 LAO 的 `0.18215` 缩放进行普通解码 |
| Plain VAE, final scale | 同一潜变量改用最终路径的 `0.1825` 缩放，辅助识别缩放影响 |
| Final Control-VAE | 原官方最终输出；其两轮 PGD、控制图、解码及保存过程保持原样 |

普通候选与最终输出共享同一次攻击得到的潜变量，不是两次独立训练。
比较的是整个最终解码路径，不能将变化直接归因于单一控制模块。
攻击代理沿用原 Notebook 的 E，R/D/S 只用于这个探索性小样本的迁移观察；后续正式验证需另留独立测试集。


## 2. 创建工作目录
下面是普通 Python 初始化代码。实验源码直接显示在后面的单元格中，无需额外上传代码文件。


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT = Path('/content/FIT5230_decoder_probe_readable_25c4201430')
SOURCE_MANIFEST_SHA256 = '25c42014300be99ae043a0eb231ff183e1c4adcbcfe6e725ed669d378ec654e6'
(PROJECT / 'm2/stealth_m2').mkdir(parents=True, exist_ok=True)
print('工作目录:', PROJECT)
(PROJECT / 'm2/stealth_m2/__init__.py').write_text('"""FIT5230 M2: resolution-aware spectral attacks and transparent evaluation."""\n\n__version__ = "0.1.0"\n')


## 3. 设置路径
选择 Colab **A100**。沿用原 Notebook 的 `CKPT_SRC`；将 `GENERATED_ROOT` 指向真实的 AI 生成图片文件夹。
不使用原流程的随机 placeholder。可提供 `REAL_ROOT` 来评估真实照片误报；空着时 FPR 显示不可计算。
图片固定随机抽取，不根据检测结果挑图。`RUN_DIR` 每次新实验使用新目录，避免覆盖记录。

`SD_MODEL_ID` 初值保留原 Notebook 使用的镜像标识；可改为你已有的相同 SD 2.1-base 本地权重路径。
记录实际来源，镜像不能自动视为与论文检查点完全一致。


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CKPT_SRC = '/content/drive/MyDrive/StealthDiffusion/checkpoints' #@param {type:"string"}
GENERATED_ROOT = '/content/drive/MyDrive/StealthDiffusion/sd_imgs' #@param {type:"string"}
REAL_ROOT = '' #@param {type:"string"}
OUTPUT_ROOT = '/content/drive/MyDrive/StealthDiffusion/M2_decoder_probe' #@param {type:"string"}
SD_MODEL_ID = 'Manojb/stable-diffusion-2-1-base' #@param {type:"string"}
N_IMAGES = 10 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
SURROGATE = 'E' #@param ['E', 'R', 'D', 'S']
ITERATIONS = 10 #@param {type:"integer"}
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = Path(OUTPUT_ROOT) / ('probe_' + stamp)
if not 1 <= N_IMAGES <= 20:
    raise ValueError('本小验证请先用 1–20 张图；默认 10 张。')
print('本次输出目录:', RUN_DIR)


### 可读的实验源码
以下每格的第一行 `%%writefile` 是 Jupyter 命令：将下面的普通 Python 原样保存成 `.py` 文件，
供原 Notebook 使用的独立 Python 3.10 环境执行。代码可以直接阅读与修改；重复运行会重新写入该模块。
新增的观察逻辑在第一个模块 `decoder_probe.py`，其余模块负责加载与评测。


#### decoder_probe.py
新增观察代码。重点看 observe_plain_decodes：从同一最终潜变量保存两张普通 VAE 候选，保留原攻击流程和随机状态。


In [ ]:
%%writefile /content/FIT5230_decoder_probe_readable_25c4201430/m2/stealth_m2/decoder_probe.py
"""Observe ordinary-VAE and final ControlVAE decoding of the same LAO latent.

This module inserts diagnostics into one immutable upstream source file. It
does not change LAO, PGD, model parameters, final decoding, or final image save
logic. Torch, NumPy and Pillow imports are delayed until observation runs.
"""

from __future__ import annotations

from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import random
import time

from .upstream_adapter import SOURCE_SHA256


SOURCE_FILE = "diff_latent_attack.py"
PROBE_SCHEMA_VERSION = 1
_START_ANCHOR = '):\n\n    if args.dataset_name == "ours_try":'
_START_INSERTION = '''    from stealth_m2.decoder_probe import (
        begin_probe as _m2_probe_begin,
        observe_plain_decodes as _m2_probe_observe,
        finish_probe as _m2_probe_finish,
    )
    _m2_decoder_probe = _m2_probe_begin(save_path, device=model.device)

'''
_FINAL_RECOMPUTATION = '''    with torch.no_grad():
        latents = torch.cat([original_latent, latent])

        for ind, t in enumerate(model.scheduler.timesteps[1 + start_step - 1:]):
            latents = diffusion_step(model, latents, context[ind], t, guidance_scale)

'''
_OBSERVE_INSERTION = '''    _m2_probe_observe(
        model.vae, latents, save_path,
        normalize=normalize, denormalize=denormalize,
        state=_m2_decoder_probe,
    )

'''
_RETURN_ANCHOR = '    return Image.fromarray(image[0]), pred_label, pred_label1, pred_label2, pred_label3, psnrv, ssimv'
_FINISH_INSERTION = '''    _m2_probe_finish(
        _m2_decoder_probe,
        final_output_path=save_path + "_adv_image.png",
        is_encoder=bool(args.is_encoder == 1),
    )

'''


def patch_probe_source(original: bytes) -> bytes:
    """Instrument only the exact pinned original; reject drift/reapplication.

    The observer runs after FINAL diffusion recomputation, not on a stale
    candidate retained from the last optimization iteration. Remove the three
    inserted statement groups and the original function AST is unchanged.
    """
    if not isinstance(original, bytes):
        raise TypeError("Original upstream source must be bytes.")
    if hashlib.sha256(original).hexdigest() != SOURCE_SHA256[SOURCE_FILE]:
        raise ValueError("Decoder probe requires the pristine pinned diff_latent_attack.py; source drift or duplicate patch rejected.")
    newline = "\r\n" if b"\r\n" in original else "\n"
    source = original.decode("utf-8").replace("\r\n", "\n")
    final_anchor = _FINAL_RECOMPUTATION + "    if args.is_encoder == 1:"
    replacements = (
        (_START_ANCHOR, "):\n\n" + _START_INSERTION + '    if args.dataset_name == "ours_try":'),
        (final_anchor, _FINAL_RECOMPUTATION + _OBSERVE_INSERTION + "    if args.is_encoder == 1:"),
        (_RETURN_ANCHOR, _FINISH_INSERTION + _RETURN_ANCHOR),
    )
    for anchor, replacement in replacements:
        if source.count(anchor) != 1:
            raise ValueError("Decoder probe anchor must occur exactly once.")
        source = source.replace(anchor, replacement, 1)
    compile(source, SOURCE_FILE, "exec")
    return source.replace("\n", newline).encode("utf-8")


def _cuda_snapshot(torch) -> list[dict]:
    """Read, never reset, CUDA profiling counters; synchronize for wall time."""
    if not torch.cuda.is_available():
        return []
    result = []
    for index in range(torch.cuda.device_count()):
        torch.cuda.synchronize(index)
        result.append({
            "device_index": index,
            "allocated_bytes": torch.cuda.memory_allocated(index),
            "reserved_bytes": torch.cuda.memory_reserved(index),
            "peak_allocated_bytes_since_external_reset": torch.cuda.max_memory_allocated(index),
            "peak_reserved_bytes_since_external_reset": torch.cuda.max_memory_reserved(index),
        })
    return result


@contextmanager
def _preserve_rng():
    """Restore Python, NumPy, torch CPU and every visible CUDA RNG on exit."""
    import numpy as np
    import torch

    python_state = random.getstate()
    numpy_state = np.random.get_state()
    devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    try:
        with torch.random.fork_rng(devices=devices, enabled=True):
            yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)


def _write_json(state: dict) -> None:
    path = Path(str(state["save_path"]) + "_decoder_probe.json")
    path.parent.mkdir(parents=True, exist_ok=True)
    public = {key: value for key, value in state.items() if not key.startswith("_")}
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(public, stream, indent=2, allow_nan=False)
        stream.write("\n")
    temporary.replace(path)


def begin_probe(save_path: str, device=None) -> dict:
    """Start per-image timing without resetting memory counters or RNGs."""
    import torch

    memory = _cuda_snapshot(torch)
    return {
        "schema_version": PROBE_SCHEMA_VERSION,
        "method": "decoder_probe",
        "observation_only": True,
        "upstream_source_sha256": SOURCE_SHA256[SOURCE_FILE],
        "save_path": str(save_path),
        "device": str(device),
        "status": "started",
        "_started_monotonic": time.perf_counter(),
        "cuda_memory_at_entry": memory,
        "memory_measurement": "Cumulative CUDA peaks since the last external reset; observer never resets counters. Peaks include diagnostic allocations.",
        "timing_measurement": "Synchronized wall time. Instrumented total includes observation; estimated base time subtracts measured observer time but is not an uninstrumented benchmark.",
        "extra_classifier_calls": 0,
    }


def _tensor_sha256(tensor) -> str:
    import torch

    value = tensor.detach().contiguous().cpu()
    digest = hashlib.sha256()
    digest.update(str(tuple(value.shape)).encode("ascii"))
    digest.update(str(value.dtype).encode("ascii"))
    digest.update(value.view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()


def observe_plain_decodes(
    vae,
    latents,
    save_path: str,
    *,
    normalize=None,
    denormalize=None,
    include_final_scale: bool | None = None,
    state: dict | None = None,
) -> dict:
    """Save ordinary VAE decodes from the exact final recomputed latent.

    Upstream has a two-image latent batch: reference at index 0, optimized at
    index 1. Both observation scales use a detached copy of that same batch.
    Each decode gets the same preserved RNG state, even for a stochastic mock.
    Observations run under no_grad; input latent storage and .grad are untouched.

    Optional normalization callbacks are the existing upstream callbacks. Their
    normalize/denormalize round trip is retained before uint8 truncation, matching
    the final output's PNG pixel conversion. No classifiers are invoked here.
    """
    import numpy as np
    from PIL import Image
    import torch

    if not isinstance(latents, torch.Tensor) or latents.ndim != 4 or latents.shape[0] != 2:
        raise ValueError("The pinned decoder probe requires the [reference, optimized] latent batch, shape [2,C,H,W].")
    if not latents.is_floating_point() or not torch.isfinite(latents).all().item():
        raise ValueError("Latents must be finite floating-point values.")
    if (normalize is None) != (denormalize is None):
        raise ValueError("Supply both upstream normalization callbacks or neither.")
    if include_final_scale is None:
        setting = os.environ.get("STEALTH_M2_PROBE_FINAL_SCALE", "1")
        if setting not in ("0", "1"):
            raise ValueError("STEALTH_M2_PROBE_FINAL_SCALE must be 0 or 1.")
        include_final_scale = setting == "1"
    if not isinstance(include_final_scale, bool):
        raise TypeError("include_final_scale must be bool.")
    if state is None:
        state = begin_probe(save_path, device=latents.device)
    if state["save_path"] != str(save_path):
        raise ValueError("Probe state and output prefix do not match.")
    specifications = [("plain_vae", 0.18215, Path(str(save_path) + "_plain_vae.png"))]
    if include_final_scale:
        specifications.append(("plain_vae_final_scale", 0.1825, Path(str(save_path) + "_plain_vae_final_scale.png")))
    if any(path.exists() for _, _, path in specifications):
        raise FileExistsError("Refusing to overwrite existing decoder-probe images.")

    memory_before = _cuda_snapshot(torch)
    started = time.perf_counter()
    was_grad_enabled = torch.is_grad_enabled()
    # Both candidates are derived from this copy after the final denoising pass.
    with torch.no_grad():
        snapshot = latents.detach().clone()
        latent_sha = _tensor_sha256(snapshot)
        candidates = {}
        for name, scale, path in specifications:
            # Restoration per candidate makes stochastic decodes comparable and
            # leaves the subsequent upstream ControlVAE path's RNG unchanged.
            with _preserve_rng(), torch.no_grad():
                decoded = vae.decode((1 / scale) * snapshot)["sample"][1:]
                if decoded.ndim != 4 or decoded.shape[0] != 1 or decoded.shape[1] != 3:
                    raise ValueError("Ordinary VAE must produce one optimized RGB image after [1:].")
                if not torch.isfinite(decoded).all().item():
                    raise ValueError("Ordinary VAE produced nonfinite pixels.")
                rgb = (decoded / 2 + 0.5).clamp(0, 1)
                if normalize is not None:
                    rgb = denormalize(normalize(rgb))
                pixels = np.uint8(rgb.detach().cpu().numpy()[0].transpose((1, 2, 0)) * 255.0)
                path.parent.mkdir(parents=True, exist_ok=True)
                Image.fromarray(pixels).save(path, format="PNG")
                candidates[name] = {
                    "path": str(path), "decode_scale": scale,
                    "latent_sha256": latent_sha,
                    "latent_selection": "sample[1:] after final diffusion recomputation",
                    "width": int(pixels.shape[1]), "height": int(pixels.shape[0]),
                    "image_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                }
    memory_after = _cuda_snapshot(torch)
    elapsed = time.perf_counter() - started
    state.update({
        "status": "observed_waiting_for_final_output",
        "latent_sha256": latent_sha,
        "latent_shape": list(snapshot.shape),
        "latent_dtype": str(snapshot.dtype),
        "latent_stage": "final recomputed diffusion output before upstream args.is_encoder branch",
        "candidates": candidates,
        "observer_seconds": elapsed,
        "observer_includes": "Two plain VAE decodes when scale control enabled; detached latent copy/hash; PNG conversion and disk writes; synchronization. No classifier calls.",
        "observer_grad_enabled": False,
        "caller_grad_enabled": was_grad_enabled,
        "rng_preservation": ["python", "numpy", "torch_cpu", "all_visible_cuda"],
        "png_conversion": "upstream normalize/denormalize roundtrip then uint8 truncation" if normalize is not None else "RGB clamp then uint8 truncation",
        "cuda_memory_before_observer": memory_before,
        "cuda_memory_after_observer": memory_after,
    })
    _write_json(state)
    return state


def finish_probe(state: dict, *, final_output_path: str, is_encoder: bool) -> dict:
    """Record final output provenance and timing, without another model call."""
    import torch

    memory = _cuda_snapshot(torch)
    total = time.perf_counter() - state["_started_monotonic"]
    path = Path(final_output_path)
    if not path.is_file():
        raise FileNotFoundError("The unchanged upstream final output was not saved: " + str(path))
    state.update({
        "status": "complete",
        "final_output": {"path": str(path), "image_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                         "decoder": "ControlVAE" if is_encoder else "ordinary_vae",
                         "decode_scale": 0.1825 if is_encoder else 0.18215,
                         "latent_sha256": state["latent_sha256"]},
        "instrumented_total_seconds": total,
        "estimated_base_seconds_excluding_observer": max(0.0, total - state["observer_seconds"]),
        "cuda_memory_at_exit": memory,
    })
    _write_json(state)
    return state


#### upstream_adapter.py
官方代码与权重准备：固定版本、单 GPU 兼容、检测器加载及原攻击启动参数。


In [ ]:
%%writefile /content/FIT5230_decoder_probe_readable_25c4201430/m2/stealth_m2/upstream_adapter.py
"""Pinned official baseline preparation and explicitly oriented detector logits.

Preparation uses only the standard library. Torch, image libraries and upstream
model code are imported only by the runtime functions that need them.
"""
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

UPSTREAM_URL = "https://github.com/wyczzy/StealthDiffusion.git"
UPSTREAM_COMMIT = "a156fb0792732e1aabc5b94a0cbecf6829e79d73"
REQUIRED_WEIGHTS = (
    "efficientnet-b0.pth", "resnet50.pth", "deit.pth", "swin-t.pth",
    "Controlvae.pt", "dncnn_color_blind.pth",
)
DETECTOR_WEIGHTS = dict(zip(("E", "R", "D", "S"), REQUIRED_WEIGHTS[:4]))
SOURCE_SHA256 = {
    "main.py": "fdf38dd8ed2700ee15a0d7d6c704465d3583462b6022e6f8ccc2af02d07fb0e9",
    "other_attacks.py": "cb46e1185518527d87a9f3f8f978288110773206277174dbb49c98c8b18c4a15",
    "demo_dir.py": "a432234d1c81b0ef7111b3ba5c2215cc2ec5f0f049858720800cd1712ab4c48b",
    "train_vae.py": "3b33d1baebe5b2c2ce180b08d320dbc672b9a90765012fbc74a231e2cb152fc3",
    "networks/trainer.py": "e4326d0d696e6c8aa09ed9b24d84d84e8605dcc68a1db252d222e9ebcd8934ff",
    "models/network_dncnn.py": "7edf9840e29c95c9eb23e79c7d856217b87c073fecd6fbc51a3a59183bcb455a",
    "diff_latent_attack.py": "5c5f776fcf3776e118f1ab534745bd668c7602cbcc06121c44f3ceede9a68851",
    "network.py": "47f309af42b9f4e0e3802e329b84c407df0e724c4ea7d28dfc2978d2f99733f0",
    "ControlVAE.py": "555fb5e094f1dbfd3d3e83e80b55e811eafb433ae34faa34b52701d7004bb9eb",
    "data/datasets.py": "11ffa483ff32f74f13aba3cc77c553b08a8aedc57642bb6bb72a097fca19e3fd",
}

# Exact replacements only. Never patch the LAO, PGD or Control-VAE algorithms.
PATCHES = {
    "main.py": [
        ('os.environ["http_proxy"] = "http://10.24.116.74:7890"\n', ""),
        ('os.environ["https_proxy"] = "http://10.24.116.74:7890"\n', ""),
        ('# os.environ["https_proxy"] = "https://127.0.0.1:7890"\n', ""),
        ("seed_torch(42)", 'seed_torch(int(os.environ.get("STEALTH_M2_SEED", "42")))'),
        ("os.makedirs(save_dir, exist_ok=True)", "os.makedirs(save_dir, exist_ok=False)"),
    ],
    "other_attacks.py": [("os.environ['CUDA_VISIBLE_DEVICES'] = \"1\"", "os.environ['CUDA_VISIBLE_DEVICES'] = \"0\""),
        ('EfficientNet.from_pretrained("efficientnet-b0", num_classes=1, image_size=None, )',
         'EfficientNet.from_name("efficientnet-b0", num_classes=1, image_size=None)'),
        ("timm.create_model('swin_base_patch4_window7_224', pretrained=True)",
         "timm.create_model('swin_base_patch4_window7_224', pretrained=False)"),
    ],
    "demo_dir.py": [("default='cuda:1'", "default='cuda:0'")],
    "train_vae.py": [('os.environ["all_proxy"] = "http://10.24.116.74:7890"\n', "")],
    "networks/trainer.py": [
        ('os.environ["http_proxy"] = "http://127.0.0.1:7890"\n', ""),
        ('os.environ["https_proxy"] = "http://127.0.0.1:7890"\n', ""),
    ],
    "models/network_dncnn.py": [('device="cuda:2"', 'device="cuda:0"')],
}


def sha256_file(path: Path | str) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def patch_source(relative_path: str, original: bytes) -> bytes:
    """Reject source drift before applying reviewed, exact infrastructure edits."""
    expected = SOURCE_SHA256.get(relative_path)
    if expected is None or hashlib.sha256(original).hexdigest() != expected:
        raise ValueError(f"Unrecognized upstream source: {relative_path}")
    # Preserve original line endings for unchanged files, including CRLF files.
    if relative_path not in PATCHES:
        return original
    newline = "\r\n" if b"\r\n" in original else "\n"
    text = original.decode("utf-8").replace("\r\n", "\n")
    for before, after in PATCHES[relative_path]:
        if text.count(before) != 1:
            raise ValueError(f"Patch anchor must occur exactly once: {relative_path}: {before!r}")
        text = text.replace(before, after, 1)
    compile(text, relative_path, "exec")  # Syntax validation; no execution.
    return text.replace("\n", newline).encode("utf-8")


def verify_weight_files(checkpoint_dir: Path | str) -> dict:
    checkpoint_dir = Path(checkpoint_dir).resolve()
    missing = [name for name in REQUIRED_WEIGHTS
               if not (checkpoint_dir / name).is_file() or (checkpoint_dir / name).stat().st_size == 0]
    if missing:
        raise FileNotFoundError("Missing or empty official checkpoints: " + ", ".join(missing))
    return {name: {"path": str((checkpoint_dir / name).resolve()),
                   "size_bytes": (checkpoint_dir / name).stat().st_size,
                   "sha256": sha256_file(checkpoint_dir / name)} for name in REQUIRED_WEIGHTS}


def prepare_upstream(source: Path | str, destination: Path | str,
                     checkpoint_dir: Path | str) -> dict:
    """Copy a pristine pinned checkout and link all six user-supplied checkpoints."""
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if destination.exists():
        raise FileExistsError(f"Refusing to replace existing runtime: {destination}")
    commit = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()
    if commit != UPSTREAM_COMMIT:
        raise ValueError(f"Expected commit {UPSTREAM_COMMIT}; got {commit}")
    dirty = subprocess.check_output(["git", "-C", str(source), "status", "--porcelain"], text=True)
    if dirty:
        raise ValueError("Official reference checkout must be pristine, including no untracked files")
    weights = verify_weight_files(checkpoint_dir)
    patched = {name: patch_source(name, (source / name).read_bytes()) for name in SOURCE_SHA256}
    tracked = subprocess.check_output(["git", "-C", str(source), "ls-files", "-z"]).decode().split("\0")
    tracked = [name for name in tracked if name]
    # All validation above is completed before creating the runtime.
    destination.mkdir(parents=True, exist_ok=False)
    file_hashes = {}
    for name in tracked:
        out = destination / name
        out.parent.mkdir(parents=True, exist_ok=True)
        data = patched[name] if name in patched else (source / name).read_bytes()
        out.write_bytes(data)
        file_hashes[name] = hashlib.sha256(data).hexdigest()
    (destination / "checkpoints").mkdir()
    for name, record in weights.items():
        (destination / "checkpoints" / name).symlink_to(record["path"])
    manifest = {"upstream_url": UPSTREAM_URL, "upstream_commit": commit,
                "runtime_files_sha256": file_hashes, "checkpoints": weights,
                "patched_files": list(PATCHES), "method": "official_baseline",
                "eot_lao_implemented": False,
                "raw_positive_class": "real", "training_class_to_idx": {"ai": 0, "nature": 1}}
    (destination / "M2_PREPARATION.json").write_text(json.dumps(manifest, indent=2) + "\n")
    return manifest


def verify_prepared_runtime(runtime: Path | str) -> dict:
    runtime = Path(runtime).resolve()
    manifest = json.loads((runtime / "M2_PREPARATION.json").read_text())
    if manifest.get("upstream_commit") != UPSTREAM_COMMIT:
        raise ValueError("Prepared runtime has a different upstream commit")
    if not set(SOURCE_SHA256).issubset(manifest.get("runtime_files_sha256", {})):
        raise ValueError("Prepared runtime manifest is incomplete")
    for name, expected in manifest["runtime_files_sha256"].items():
        if Path(name).is_absolute() or ".." in Path(name).parts:
            raise ValueError("Invalid runtime manifest path")
        if sha256_file(runtime / name) != expected:
            raise ValueError(f"Prepared runtime has been modified: {name}")
    weights = verify_weight_files(runtime / "checkpoints")
    for name in REQUIRED_WEIGHTS:
        if weights[name]["sha256"] != manifest["checkpoints"][name]["sha256"]:
            raise ValueError(f"Checkpoint changed since preparation: {name}")
    return manifest


def make_fake_logit_adapter(model, *, input_size: int = 224, raw_positive_class: str = "real"):
    """Wrap an official scalar head; input is unnormalized RGB BCHW in [0,1]."""
    import torch
    from torch import nn
    from torch.nn import functional as F

    if raw_positive_class not in {"real", "fake"}:
        raise ValueError("raw_positive_class must be explicitly real or fake")
    if input_size <= 0:
        raise ValueError("input_size must be positive")

    class FakeLogitAdapter(nn.Module):
        def __init__(self):
            super().__init__()
            self.model = model.eval().requires_grad_(False)
            self.input_size = input_size
            self.raw_positive_class = raw_positive_class
            self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
            self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        def forward(self, images):
            if images.ndim != 4 or images.shape[1] != 3 or not images.is_floating_point():
                raise ValueError("Expected floating RGB tensor of shape (B,3,H,W)")
            if images.shape[0] == 0 or min(images.shape[-2:]) < 1:
                raise ValueError("Empty image batches or spatial dimensions are unsupported")
            if not torch.isfinite(images).all() or images.min() < 0 or images.max() > 1:
                raise ValueError("Detector input must be finite RGB in [0,1]")
            if images.shape[-2:] != (self.input_size, self.input_size):
                images = F.interpolate(images, size=(self.input_size, self.input_size),
                                       mode="bilinear", align_corners=False, antialias=True)
            raw = self.model((images - self.mean) / self.std)
            if not isinstance(raw, torch.Tensor) or raw.shape != (images.shape[0], 1):
                raise ValueError("Official scalar detector must return shape (B,1)")
            if not torch.isfinite(raw).all():
                raise ValueError("Non-finite detector logits")
            return raw[:, 0] * (-1 if self.raw_positive_class == "real" else 1)

    return FakeLogitAdapter()


def load_official_detector(upstream_dir: Path | str, checkpoint_dir: Path | str,
                          name: str, device: str = "cuda:0", *, input_size: int = 224,
                          raw_positive_class: str = "real"):
    """Load only official scalar heads, strictly, without pretrained downloads.

    Checkpoint provenance/class polarity must be calibrated using separate known
    real/fake images by the caller before attack or final test evaluation.
    """
    import torch

    aliases = {"efficientnet-b0": "E", "resnet50": "R", "deit": "D", "swin-t": "S"}
    name = aliases.get(name, name.upper())
    if name not in DETECTOR_WEIGHTS:
        raise ValueError(f"Unknown official detector: {name}")
    if input_size != 224:
        raise ValueError("Official evaluation uses input_size=224; native input may be any size")
    upstream_dir = Path(upstream_dir).resolve()
    if sha256_file(upstream_dir / "network.py") != SOURCE_SHA256["network.py"]:
        raise ValueError("Unrecognized official ResNet implementation")
    checkpoint = Path(checkpoint_dir).resolve() / DETECTOR_WEIGHTS[name]
    if not checkpoint.is_file() or checkpoint.stat().st_size == 0:
        raise FileNotFoundError(f"Missing/empty detector checkpoint: {checkpoint}")
    if name == "R":
        spec = importlib.util.spec_from_file_location("_stealth_official_network", upstream_dir / "network.py")
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        model = module.resnet50(num_classes=1)
    elif name == "E":
        from efficientnet_pytorch import EfficientNet
        model = EfficientNet.from_name("efficientnet-b0", num_classes=1, image_size=None)
    else:
        import timm
        architecture, width = (("deit_base_patch16_224", 768) if name == "D" else
                               ("swin_base_patch4_window7_224", 1024))
        model = timm.create_model(architecture, pretrained=False)
        model.head = torch.nn.Linear(width, 1)
    payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or not isinstance(payload.get("model"), dict):
        raise ValueError("Expected official checkpoint with a 'model' state dictionary")
    model.load_state_dict(payload["model"], strict=True)
    adapter = make_fake_logit_adapter(model, input_size=input_size, raw_positive_class=raw_positive_class)
    adapter.checkpoint_sha256 = sha256_file(checkpoint)
    adapter.detector_name = name
    return adapter.to(device).eval()


def baseline_launch_plan(runtime: Path | str, images_dir: Path | str, output_dir: Path | str,
                         *, surrogate: str = "S", seed: int = 42, iterations: int = 10,
                         sd_model_id: str | None = None, python: str = sys.executable,
                         method: str = "official_baseline", res: int = 224) -> dict:
    """Build a checked launch plan. Run separately via execute_baseline_plan."""
    from PIL import Image

    if method != "official_baseline":
        raise NotImplementedError("EOT LAO is not implemented by the official baseline launcher")
    if res != 224:
        raise ValueError("Unchanged official baseline supports res=224 only; 512 needs an explicit adapter experiment")
    if surrogate not in DETECTOR_WEIGHTS or iterations < 1 or not (0 <= seed < 2**32):
        raise ValueError("Invalid surrogate, iterations, or seed")
    runtime, images_dir, output_dir = map(lambda p: Path(p).resolve(), (runtime, images_dir, output_dir))
    if output_dir.exists() or "{" in str(output_dir) or "}" in str(output_dir):
        raise FileExistsError("Output must be a new concrete directory, without format placeholders")
    preparation = verify_prepared_runtime(runtime)
    if not images_dir.is_dir():
        raise FileNotFoundError(f"Input image directory not found: {images_dir}")
    entries = list(images_dir.iterdir())
    if not entries or any(not p.is_file() or p.suffix.lower() not in {".png", ".jpg", ".jpeg"} for p in entries):
        raise ValueError("Input must be a nonempty flat directory containing only PNG/JPEG images")
    from natsort import natsorted, ns
    # Use exactly the ordering used by upstream main.py, including case/path handling.
    entries = natsorted(entries, key=str, alg=ns.PATH)
    records = []
    for index, path in enumerate(entries):
        with Image.open(path) as image:
            size = list(image.size)
            image.verify()
        records.append({"source": str(path), "sha256": sha256_file(path), "source_width_height": size,
                        "expected_output": f"{index:04d}_adv_image.png"})
    order = [surrogate] + [n for n in ("E", "R", "D", "S") if n != surrogate]
    command = [python, "main.py", "--model_name", ",".join(order), "--images_root", str(images_dir),
               "--save_dir", str(output_dir), "--encoder_weights", str(runtime / "checkpoints/Controlvae.pt"),
               "--iterations", str(iterations), "--diffusion_steps", "20", "--start_step", "18", "--res", "224"]
    if sd_model_id:
        command.extend(["--pretrained_diffusion_path", sd_model_id])
    return {"command": command, "cwd": str(runtime), "output_dir": str(output_dir),
            "environment": {"CUDA_VISIBLE_DEVICES": "0", "PYTHONHASHSEED": str(seed),
                            "STEALTH_M2_SEED": str(seed), "CUBLAS_WORKSPACE_CONFIG": ":4096:8"},
            "upstream_commit": UPSTREAM_COMMIT, "method": method, "seed": seed,
            "iterations": iterations, "surrogate": surrogate, "images_dir": str(images_dir),
            "sd_model_id": sd_model_id or "stabilityai/stable-diffusion-2-1-base",
            "diffusion_input_resolution": 224, "diffusion_output_resolution": 224,
            "detector_input_resolution": 224, "control_vae_sample_size_metadata": 512,
            "input_records": records, "checkpoints": preparation["checkpoints"],
            "eot_lao_implemented": False}


def execute_baseline_plan(plan: dict) -> dict:
    """Execute the reviewed official launch and check that every expected image exists."""
    import torch
    from PIL import Image

    if not torch.cuda.is_available():
        raise RuntimeError("Official baseline requires a CUDA GPU; no CPU/MPS fallback")
    if Path(plan["output_dir"]).exists():
        raise FileExistsError("Refusing to overwrite an existing experiment")
    verify_prepared_runtime(plan["cwd"])
    current_inputs = {str(p.resolve()) for p in Path(plan["images_dir"]).iterdir()}
    if current_inputs != {record["source"] for record in plan["input_records"]}:
        raise ValueError("Input directory contents changed after planning")
    for record in plan["input_records"]:
        if sha256_file(record["source"]) != record["sha256"]:
            raise ValueError(f"Input changed after planning: {record['source']}")
    env = os.environ.copy()
    env.update(plan["environment"])
    # These are launcher environment settings, not a claim that arbitrary GPU
    # libraries produce bitwise identical results on different versions/devices.
    subprocess.run(plan["command"], cwd=plan["cwd"], env=env, check=True)
    output = Path(plan["output_dir"])
    for record in plan["input_records"]:
        generated = output / record["expected_output"]
        with Image.open(generated) as image:
            if image.size != (224, 224):
                raise ValueError(f"Unexpected official output dimensions: {image.size}")
            image.verify()
        record["output_sha256"] = sha256_file(generated)
    with (output / "M2_RUN.json").open("x") as handle:
        json.dump(plan, handle, indent=2)
        handle.write("\n")
    return plan


#### data.py
图像读取辅助函数。此实验使用其中的 RGB 图片读取接口。


In [ ]:
%%writefile /content/FIT5230_decoder_probe_readable_25c4201430/m2/stealth_m2/data.py
"""Portable image manifests with an explicit train/development/test boundary."""

from __future__ import annotations

import csv
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import torch
from PIL import Image, ImageOps


SUPPORTED_SUFFIXES = frozenset({".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"})
SPLITS = frozenset({"train", "dev", "test"})


@dataclass(frozen=True)
class ImageRecord:
    image_id: str
    path: Path
    label: int  # 0 = real, 1 = generated
    split: str
    source: str = ""
    sha256: str = ""  # Hash of oriented RGB pixels and dimensions, not metadata.


def _open_rgb(path: Path) -> Image.Image:
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


def _pixel_hash(image: Image.Image) -> str:
    digest = hashlib.sha256()
    digest.update(f"RGB:{image.width}:{image.height}:".encode("ascii"))
    digest.update(image.tobytes())
    return digest.hexdigest()


def load_manifest(
    csv_path: str | Path,
    root: str | Path | None = None,
    required_splits: Iterable[str] = (),
) -> list[ImageRecord]:
    """Validate a CSV and return resolved image records.

    Required columns: image_id,path,label,split. Optional column: source.
    Paths must be relative to ``root`` (the CSV directory by default), so the
    same manifest can be uploaded to Colab. Image IDs are globally unique.
    Exact decoded-image duplicates across splits are rejected even if file
    names, image metadata, or lossless encoding differ. This is not a near-
    duplicate or subject-overlap detector; group related images before splitting.
    """
    csv_path = Path(csv_path).expanduser().resolve()
    base = Path(root).expanduser().resolve() if root is not None else csv_path.parent
    requested = set(required_splits)
    if requested - SPLITS:
        raise ValueError(f"Unknown required splits: {sorted(requested - SPLITS)}")
    records: list[ImageRecord] = []
    seen_ids: set[str] = set()
    seen_pixels: dict[str, tuple[str, str, int]] = {}
    with csv_path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        missing = {"image_id", "path", "label", "split"} - set(reader.fieldnames or ())
        if missing:
            raise ValueError(f"Manifest is missing columns: {sorted(missing)}")
        for line, row in enumerate(reader, start=2):
            image_id = (row.get("image_id") or "").strip()
            if not image_id or image_id in seen_ids:
                raise ValueError(f"Line {line}: empty or duplicate image_id {image_id!r}")
            label_text = (row.get("label") or "").strip()
            if label_text not in {"0", "1"}:
                raise ValueError(f"Line {line}: label must be 0 (real) or 1 (generated)")
            label = int(label_text)
            split = (row.get("split") or "").strip()
            if split not in SPLITS:
                raise ValueError(f"Line {line}: split must be train, dev, or test")
            path_text = (row.get("path") or "").strip()
            relative = Path(path_text)
            if not path_text or relative.is_absolute():
                raise ValueError(f"Line {line}: path must be a nonempty relative image path")
            path = (base / relative).resolve()
            if path.suffix.lower() not in SUPPORTED_SUFFIXES:
                raise ValueError(f"Line {line}: unsupported image extension: {path.suffix}")
            if not path.is_file():
                raise FileNotFoundError(f"Line {line}: image does not exist: {path}")
            try:
                digest = _pixel_hash(_open_rgb(path))
            except (OSError, ValueError) as exc:
                raise ValueError(f"Line {line}: cannot decode image: {path}") from exc
            previous = seen_pixels.get(digest)
            if previous is not None:
                old_id, old_split, old_label = previous
                if old_split != split:
                    raise ValueError(
                        f"Image content overlaps splits: {old_id} ({old_split}) and "
                        f"{image_id} ({split})"
                    )
                if old_label != label:
                    raise ValueError(f"Identical image has conflicting labels: {old_id}, {image_id}")
            seen_pixels[digest] = (image_id, split, label)
            seen_ids.add(image_id)
            records.append(ImageRecord(image_id, path, label, split, (row.get("source") or "").strip(), digest))
    if not records:
        raise ValueError("Manifest contains no image rows")
    absent = requested - {record.split for record in records}
    if absent:
        raise ValueError(f"Manifest has no images in required splits: {sorted(absent)}")
    return records


def select_records(
    records: Iterable[ImageRecord], split: str, label: int | None = None
) -> list[ImageRecord]:
    """Explicitly select a split; no implicit train/test fallback is permitted."""
    if split not in SPLITS:
        raise ValueError(f"Unknown split: {split}")
    if label is not None and label not in (0, 1):
        raise ValueError("label must be 0, 1, or None")
    return [record for record in records if record.split == split and (label is None or record.label == label)]


def load_rgb(
    record_or_path: ImageRecord | str | Path,
    size: int | tuple[int, int] | None = None,
) -> torch.Tensor:
    """Read oriented RGB as float32 CHW in [0,1]; tuple size is (height,width)."""
    path = record_or_path.path if isinstance(record_or_path, ImageRecord) else Path(record_or_path)
    image = _open_rgb(path)
    if size is not None:
        height, width = (size, size) if isinstance(size, int) else size
        if height < 1 or width < 1:
            raise ValueError("Image dimensions must be positive")
        image = image.resize((width, height), Image.Resampling.BICUBIC)
    pixels = np.asarray(image, dtype=np.float32).copy() / 255.0
    return torch.from_numpy(pixels).permute(2, 0, 1).contiguous()


#### evaluation.py
评测公式：条件攻击成功率、真实图误报率、PSNR、SSIM 和实际扰动。


In [ ]:
%%writefile /content/FIT5230_decoder_probe_readable_25c4201430/m2/stealth_m2/evaluation.py
"""Real postprocessing, image quality, and paired attack-effect accounting.

Detector scores are fake logits: score >= threshold means generated. Conditional
ASR uses only generated images detected BEFORE attack under the SAME detector
and postprocessing condition. Unconditional escape and real-image FPR are named
separately to avoid crediting attacks for pre-existing detector failures.
"""

from __future__ import annotations

import io
import math
from collections import defaultdict
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from .data import ImageRecord


CONDITIONS = ("none", "jpeg", "resize", "jpeg_resize")


def _batch(images: torch.Tensor) -> tuple[torch.Tensor, bool]:
    single = images.ndim == 3
    batch = images.unsqueeze(0) if single else images
    if batch.ndim != 4 or batch.shape[1] != 3 or min(batch.shape) < 1:
        raise ValueError("Expected nonempty RGB CHW or BCHW tensor")
    if not batch.is_floating_point() or not torch.isfinite(batch).all():
        raise ValueError("Images must contain finite floating-point values")
    if batch.min() < 0 or batch.max() > 1:
        raise ValueError("Images must be in [0,1]")
    return batch, single


def apply_condition(
    images: torch.Tensor,
    condition: str,
    jpeg_quality: int = 75,
    resize_scale: float = 0.5,
) -> torch.Tensor:
    """Apply actual Pillow operations, preserving tensor shape/device/dtype.

    JPEG is an in-memory encode/decode, not a differentiable approximation.
    Resize is bicubic downsampling followed by bicubic return to original size.
    ``jpeg_resize`` applies JPEG first, then resize. Active conditions use the
    8-bit RGB pixels a real image pipeline receives. ``none`` preserves input
    values exactly. These functions are evaluation-only and detach gradients.
    """
    batch, single = _batch(images)
    if condition not in CONDITIONS:
        raise ValueError(f"Unknown condition {condition!r}; choose from {CONDITIONS}")
    if not isinstance(jpeg_quality, int) or not 1 <= jpeg_quality <= 100:
        raise ValueError("jpeg_quality must be an integer in [1,100]")
    if not math.isfinite(resize_scale) or not 0 < resize_scale <= 1:
        raise ValueError("resize_scale must be in (0,1]")
    if condition == "none":
        return images.detach().clone()
    result = []
    for tensor in batch.detach().cpu():
        pixels = tensor.permute(1, 2, 0).float().mul(255).round().to(torch.uint8).numpy()
        image = Image.fromarray(pixels)
        if condition in {"jpeg", "jpeg_resize"}:
            buffer = io.BytesIO()
            image.save(buffer, format="JPEG", quality=jpeg_quality, subsampling=2)
            buffer.seek(0)
            with Image.open(buffer) as decoded:
                image = decoded.convert("RGB")
        if condition in {"resize", "jpeg_resize"}:
            width, height = image.size
            small = (max(1, round(width * resize_scale)), max(1, round(height * resize_scale)))
            image = image.resize(small, Image.Resampling.BICUBIC).resize((width, height), Image.Resampling.BICUBIC)
        array = np.asarray(image, dtype=np.float32).copy() / 255.0
        result.append(torch.from_numpy(array).permute(2, 0, 1))
    output = torch.stack(result).to(device=images.device, dtype=images.dtype)
    return output[0] if single else output


def image_quality(clean: torch.Tensor, attacked: torch.Tensor) -> list[dict[str, float]]:
    """Per-image PSNR, Gaussian SSIM, and L-infinity before postprocessing.

    SSIM uses an 11x11 Gaussian (sigma 1.5), valid windows, population moments,
    K1=.01 and K2=.03, averaged over RGB. For images smaller than 11 pixels the
    largest fitting odd window is used. Exact equality has PSNR = +infinity;
    aggregation reports null for an unbounded mean and counts exact matches.
    """
    original, _ = _batch(clean)
    adversarial, _ = _batch(attacked)
    if original.shape != adversarial.shape:
        raise ValueError("Clean and attacked images must have the same shape")
    x, y = original.detach().float().cpu(), adversarial.detach().float().cpu()
    mse = (x - y).square().flatten(1).mean(1)
    linf = (x - y).abs().flatten(1).amax(1)
    width = min(11, x.shape[-2], x.shape[-1])
    width -= 1 - width % 2
    coordinate = torch.arange(width, dtype=x.dtype) - (width - 1) / 2
    gaussian = torch.exp(-coordinate.square() / (2 * 1.5**2))
    gaussian = gaussian / gaussian.sum()
    kernel = torch.outer(gaussian, gaussian).view(1, 1, width, width).repeat(3, 1, 1, 1)
    smooth = lambda value: F.conv2d(value, kernel, groups=3)
    mu_x, mu_y = smooth(x), smooth(y)
    variance_x = (smooth(x.square()) - mu_x.square()).clamp_min(0)
    variance_y = (smooth(y.square()) - mu_y.square()).clamp_min(0)
    covariance = smooth(x * y) - mu_x * mu_y
    ssim = ((2 * mu_x * mu_y + 0.01**2) * (2 * covariance + 0.03**2)) / (
        (mu_x.square() + mu_y.square() + 0.01**2) * (variance_x + variance_y + 0.03**2)
    )
    ssim = ssim.flatten(1).mean(1)
    return [
        {"psnr": math.inf if error == 0 else -10 * math.log10(error), "ssim": float(score), "linf": float(distance)}
        for error, score, distance in zip(mse.tolist(), ssim.tolist(), linf.tolist())
    ]


def _scores(values: Sequence[float] | torch.Tensor, count: int, name: str) -> list[float]:
    tensor = torch.as_tensor(values).detach().cpu()
    if tensor.ndim != 1 or len(tensor) != count:
        raise ValueError(f"{name} must have shape ({count},)")
    if not torch.isfinite(tensor).all():
        raise ValueError(f"{name} contains a nonfinite score")
    return [float(value) for value in tensor]


def make_prediction_records(
    records: Sequence[ImageRecord | Mapping[str, Any]],
    *,
    method: str,
    detector: str,
    condition: str,
    raw_clean_logits: Sequence[float] | torch.Tensor,
    clean_logits: Sequence[float] | torch.Tensor,
    attacked_logits: Sequence[float] | torch.Tensor | None = None,
    quality: Sequence[Mapping[str, float]] | None = None,
    seconds_per_image: Sequence[float] | None = None,
    threshold: float = 0.0,
) -> list[dict[str, Any]]:
    """Create auditable paired rows; caller performs batched detector inference.

    ``raw_clean_logits`` scores original images; ``clean_logits`` scores those
    originals after ``condition``. ``attacked_logits`` scores attacked images
    after that same condition. Omit attacked logits for real-only FPR rows.
    Quality metrics must compare original and attacked pixels before processing.
    """
    if not math.isfinite(threshold):
        raise ValueError("threshold must be finite")
    count = len(records)
    raw = _scores(raw_clean_logits, count, "raw_clean_logits")
    before = _scores(clean_logits, count, "clean_logits")
    after = [None] * count if attacked_logits is None else _scores(attacked_logits, count, "attacked_logits")
    if quality is not None and len(quality) != count:
        raise ValueError("quality must have one entry per image")
    if seconds_per_image is not None and len(seconds_per_image) != count:
        raise ValueError("seconds_per_image must have one entry per image")
    rows = []
    for index, record in enumerate(records):
        get = record.get if isinstance(record, Mapping) else lambda key, default=None: getattr(record, key, default)
        label = int(get("label"))
        if label not in (0, 1):
            raise ValueError("Labels must be 0 (real) or 1 (generated)")
        detected = before[index] >= threshold
        attacked_detected = None if after[index] is None else after[index] >= threshold
        eligible = label == 1 and detected and after[index] is not None
        row = {
            "image_id": str(get("image_id")), "label": label,
            "split": get("split", ""), "source": get("source", ""),
            "method": method, "detector": detector, "condition": condition,
            "threshold": threshold, "raw_clean_logit": raw[index],
            "clean_logit": before[index], "attacked_logit": after[index],
            "raw_clean_detected": raw[index] >= threshold, "clean_detected": detected,
            "attacked_detected": attacked_detected, "eligible": eligible,
            "attack_success": eligible and not attacked_detected,
            "unconditional_escape": None if label != 1 or attacked_detected is None else not attacked_detected,
            "psnr": None, "ssim": None, "linf": None, "seconds": None,
        }
        if quality is not None:
            for key in ("psnr", "ssim", "linf"):
                row[key] = quality[index].get(key)
        if seconds_per_image is not None:
            seconds = float(seconds_per_image[index])
            if not math.isfinite(seconds) or seconds < 0:
                raise ValueError("seconds_per_image must be finite and nonnegative")
            row["seconds"] = seconds
        rows.append(row)
    return rows


def aggregate_records(records: Iterable[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Aggregate by method/detector/condition/threshold with explicit counts.

    Empty denominators are null, never 0. Rates are proportions in [0,1]. The
    real-image FPR uses transformed clean images; real images are not attacked.
    Repeated image IDs within a group are rejected to prevent double counting.
    """
    groups: dict[tuple, list[Mapping[str, Any]]] = defaultdict(list)
    for row in records:
        groups[(row["method"], row["detector"], row["condition"], row["threshold"])].append(row)
    output = []
    for (method, detector, condition, threshold), rows in sorted(groups.items()):
        if len({row["image_id"] for row in rows}) != len(rows):
            raise ValueError(f"Duplicate image_id in group: {method}/{detector}/{condition}")
        generated = [row for row in rows if row["label"] == 1]
        attacked = [row for row in generated if row["attacked_logit"] is not None]
        real = [row for row in rows if row["label"] == 0]
        eligible = sum(bool(row["eligible"]) for row in attacked)
        successes = sum(bool(row["attack_success"]) for row in attacked)
        escapes = sum(bool(row["unconditional_escape"]) for row in attacked)
        real_fp = sum(bool(row["clean_detected"]) for row in real)
        clean_tp = sum(bool(row["clean_detected"]) for row in generated)
        summary = {
            "method": method, "detector": detector, "condition": condition, "threshold": threshold,
            "n_images": len(rows), "n_generated": len(generated), "n_generated_attacked": len(attacked),
            "clean_fake_detected_count": clean_tp, "clean_tpr": clean_tp / len(generated) if generated else None,
            "eligible_count": eligible, "success_count": successes,
            "conditional_asr": successes / eligible if eligible else None,
            "escape_count": escapes, "unconditional_escape_rate": escapes / len(attacked) if attacked else None,
            "n_real": len(real), "real_fp_count": real_fp, "real_fpr": real_fp / len(real) if real else None,
            "psnr_exact_match_count": sum(row.get("psnr") == math.inf for row in attacked),
        }
        for metric in ("psnr", "ssim", "linf", "seconds"):
            values = [float(row[metric]) for row in attacked if row.get(metric) is not None]
            summary[f"mean_{metric}"] = sum(values) / len(values) if values and all(map(math.isfinite, values)) else None
        output.append(summary)
    return output


#### decoder_probe_run.py
执行流程：选择图片、准备环境、运行官方攻击、重新读取 PNG 并汇总配对结果。


In [ ]:
%%writefile /content/FIT5230_decoder_probe_readable_25c4201430/m2/stealth_m2/decoder_probe_run.py
"""Small, observation-only decoder probe around the pinned M1 pipeline.

No new attack objective is implemented. All three saved images use the final
latent selected by the original optimization. The same-scale plain VAE output
separates the decoder comparison from the original 0.18215/0.1825 scale change.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import os
from pathlib import Path
import random
import re
import shutil
import subprocess
import sys
import tempfile
import threading
from datetime import datetime, timezone

from PIL import Image, ImageOps

from . import upstream_adapter as upstream


VARIANTS = {
    "plain_vae": "plain_vae.png",
    "plain_vae_final_scale": "plain_vae_final_scale.png",
    "control_vae": "adv_image.png",
}
PROBE_FILE = "diff_latent_attack.py"
_PLACEHOLDER = re.compile(r"placeholder|dummy|your[_ -]?(?:generated|image|path)", re.IGNORECASE)


def _now() -> str:
    return datetime.now(timezone.utc).isoformat()


def _json(path: Path, value: dict, *, replace: bool = False) -> None:
    text = json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
    if replace:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                         prefix=f".{path.name}.", delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(text)
        try:
            temporary.replace(path)
        finally:
            temporary.unlink(missing_ok=True)
    else:
        with path.open("x", encoding="utf-8") as stream:
            stream.write(text)


def _canonical_png(path: Path) -> bytes:
    with Image.open(path) as image:
        pixels = ImageOps.exif_transpose(image).convert("RGB").resize((224, 224), Image.Resampling.LANCZOS)
        clean = Image.frombytes("RGB", (224, 224), pixels.tobytes())
    stream = io.BytesIO()
    clean.save(stream, format="PNG", compress_level=9)
    return stream.getvalue()


def select_inputs(generated_root, work_dir, *, n_images: int = 10, seed: int = 42, real_root=None) -> dict:
    """Freeze real image files as canonical RGB inputs; never fabricate inputs."""
    if not isinstance(n_images, int) or isinstance(n_images, bool) or n_images < 1:
        raise ValueError("n_images must be a positive integer")
    if not isinstance(seed, int) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    work_dir = Path(work_dir).expanduser().resolve()
    if work_dir.exists():
        raise FileExistsError(f"Work directory must be new: {work_dir}")
    rng = random.Random(seed)
    seen = {}
    selected = []
    roots = [(1, Path(generated_root).expanduser().resolve())]
    if real_root is not None:
        roots.append((0, Path(real_root).expanduser().resolve()))
    for label, root in roots:
        if not root.is_dir():
            raise FileNotFoundError(f"Image root does not exist: {root}")
        if _PLACEHOLDER.search(str(root)):
            raise ValueError(f"Placeholder input path is not permitted: {root}")
        candidates = sorted(path.resolve() for path in root.rglob("*")
                            if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg"})
        if any(_PLACEHOLDER.search(path.name) for path in candidates):
            raise ValueError("Named placeholder/dummy images are not valid probe data")
        rng.shuffle(candidates)
        count = 0
        for path in candidates:
            original_hash = upstream.sha256_file(path)
            canonical = _canonical_png(path)
            canonical_hash = hashlib.sha256(canonical).hexdigest()
            if upstream.sha256_file(path) != original_hash:
                raise ValueError(f"Source image changed while selecting: {path}")
            if canonical_hash in seen:
                if seen[canonical_hash] != label:
                    raise ValueError("Same canonical image occurs in both real and generated inputs")
                continue
            seen[canonical_hash] = label
            image_id = f"{count:04d}" if label == 1 else f"real_{count:04d}"
            relative_path = f"inputs/{'generated' if label == 1 else 'real'}/{image_id}.png"
            selected.append(({"image_id": image_id, "label": label, "source_path": str(path),
                              "source_sha256": original_hash, "canonical_path": relative_path,
                              "canonical_sha256": canonical_hash}, canonical))
            count += 1
            if count == n_images:
                break
        if count != n_images:
            raise ValueError(f"Need {n_images} distinct class-{label} images; found {count} in {root}")
    manifest = {"selection_seed": seed, "n_generated": n_images,
                "n_real": n_images if real_root is not None else 0,
                "generated_root": str(roots[0][1]), "real_root": str(roots[1][1]) if len(roots) == 2 else None,
                "canonical_format": "224x224 RGB PNG, EXIF orientation, LANCZOS, metadata removed",
                "records": [record for record, _ in selected], "created_at": _now()}
    work_dir.mkdir(parents=True, exist_ok=False)
    try:
        for record, canonical in selected:
            output = work_dir / record["canonical_path"]
            output.parent.mkdir(parents=True, exist_ok=True)
            output.write_bytes(canonical)
        _json(work_dir / "input_manifest.json", manifest)
    except BaseException:
        shutil.rmtree(work_dir)
        raise
    return manifest


def verify_inputs(work_dir) -> dict:
    work_dir = Path(work_dir).resolve()
    manifest = json.loads((work_dir / "input_manifest.json").read_text())
    seen_ids, seen_hashes, expected_files = set(), set(), set()
    for row in manifest["records"]:
        relative = Path(row["canonical_path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Invalid canonical path in input manifest")
        path = work_dir / relative
        if row["image_id"] in seen_ids or row["canonical_sha256"] in seen_hashes:
            raise ValueError("Input manifest contains duplicate IDs or image content")
        if upstream.sha256_file(path) != row["canonical_sha256"]:
            raise ValueError(f"Canonical input changed: {path}")
        with Image.open(path) as image:
            if image.size != (224, 224) or image.mode != "RGB":
                raise ValueError(f"Canonical input must be 224x224 RGB: {path}")
            image.verify()
        seen_ids.add(row["image_id"])
        seen_hashes.add(row["canonical_sha256"])
        expected_files.add(path.resolve())
    actual = {path.resolve() for path in (work_dir / "inputs").rglob("*") if path.is_file()}
    if actual != expected_files:
        raise ValueError("Input folder contents changed after selection")
    for label, name in ((1, "n_generated"), (0, "n_real")):
        if sum(row["label"] == label for row in manifest["records"]) != manifest[name]:
            raise ValueError("Input manifest class counts do not match its records")
    return manifest


def prepare_probe(source, checkpoint_dir, work_dir, *, runtime_dir=None) -> dict:
    """Prepare pinned infrastructure, then apply only decoder observation hooks."""
    from .decoder_probe import patch_probe_source

    work_dir = Path(work_dir).resolve()
    verify_inputs(work_dir)
    reference_path = work_dir / "runtime_reference.json"
    if reference_path.exists():
        raise FileExistsError("Probe runtime reference already exists")
    patched = patch_probe_source((Path(source) / PROBE_FILE).read_bytes())
    runtime = Path(runtime_dir).expanduser().resolve() if runtime_dir is not None else work_dir / "runtime"
    preparation = upstream.prepare_upstream(source, runtime, checkpoint_dir)
    (runtime / PROBE_FILE).write_bytes(patched)
    preparation["runtime_files_sha256"][PROBE_FILE] = hashlib.sha256(patched).hexdigest()
    preparation["patched_files"] = sorted(set(preparation["patched_files"]) | {PROBE_FILE})
    preparation.update({"method": "decoder_probe", "observation_only": True,
                        "unchanged_official_baseline": False,
                        "decoder_probe": {
                            "instrumented_file": PROBE_FILE,
                            "original_source_sha256": upstream.SOURCE_SHA256[PROBE_FILE],
                            "variants": list(VARIANTS), "same_final_latent": True,
                            "description": "Adds plain-VAE observations beside the original final Control-VAE output; no new attack objective.",
                            "plain_vae_scale": 0.18215, "final_decoder_scale": 0.1825}})
    _json(runtime / "M2_PREPARATION.json", preparation, replace=True)
    upstream.verify_prepared_runtime(runtime)
    _json(reference_path, {"runtime_dir": str(runtime), "method": "decoder_probe", "observation_only": True,
                           "preparation_sha256": upstream.sha256_file(runtime / "M2_PREPARATION.json"),
                           "prepared_at": _now()})
    return preparation


def _runtime_path(work_dir: Path) -> Path:
    reference = json.loads((work_dir / "runtime_reference.json").read_text())
    runtime = Path(reference["runtime_dir"]).resolve()
    if upstream.sha256_file(runtime / "M2_PREPARATION.json") != reference["preparation_sha256"]:
        raise ValueError("Runtime preparation changed after the saved work-directory reference")
    return runtime


def _verify_probe_runtime(work_dir: Path) -> dict:
    preparation = upstream.verify_prepared_runtime(_runtime_path(work_dir))
    if preparation.get("method") != "decoder_probe" or preparation.get("observation_only") is not True:
        raise ValueError("Runtime is not an explicitly prepared observation-only decoder probe")
    return preparation


def _stream_command(plan: dict, log_path: Path) -> None:
    """Run with check=True while teeing all combined process output live to a log."""
    env = os.environ.copy()
    env.update(plan["environment"])
    read_fd, write_fd = os.pipe()
    with log_path.open("x", encoding="utf-8") as log:
        def relay():
            with os.fdopen(read_fd, "r", encoding="utf-8", errors="replace") as stream:
                for line in stream:
                    log.write(line)
                    log.flush()
                    print(line, end="", flush=True)
        reader = threading.Thread(target=relay, daemon=True)
        reader.start()
        try:
            subprocess.run(plan["command"], cwd=plan["cwd"], env=env,
                           stdout=write_fd, stderr=subprocess.STDOUT, check=True)
        finally:
            os.close(write_fd)
            reader.join()


def run_probe(work_dir, *, surrogate: str = "E", iterations: int = 10,
              sd_model_id: str | None = None, python: str = sys.executable) -> dict:
    """Execute once; failure/interruption is recorded and never marked successful."""
    import torch
    from . import decoder_probe

    work_dir = Path(work_dir).resolve()
    metadata_path = work_dir / "run_metadata.json"
    if metadata_path.exists() or (work_dir / "outputs").exists() or (work_dir / "run.log").exists():
        raise FileExistsError("Probe run artifacts already exist; use a fresh work directory")
    selection = verify_inputs(work_dir)
    preparation = _verify_probe_runtime(work_dir)
    metadata = {"status": "preparing", "method": "decoder_probe", "observation_only": True,
                "unchanged_official_baseline": False, "started_at": _now(),
                "input_manifest": selection, "input_manifest_sha256": upstream.sha256_file(work_dir / "input_manifest.json"),
                "preparation": preparation, "surrogate": surrogate,
                "runner_sha256": upstream.sha256_file(__file__),
                "probe_helper_sha256": upstream.sha256_file(decoder_probe.__file__)}
    _json(metadata_path, metadata)
    try:
        if not torch.cuda.is_available():
            raise RuntimeError("Decoder probe requires a CUDA GPU; no CPU/MPS fallback")
        plan = upstream.baseline_launch_plan(_runtime_path(work_dir), work_dir / "inputs/generated", work_dir / "outputs",
                                              surrogate=surrogate, seed=selection["selection_seed"],
                                              iterations=iterations, sd_model_id=sd_model_id, python=python)
        plan.update({"method": "decoder_probe", "observation_only": True,
                     "unchanged_official_baseline": False, "variants": list(VARIANTS)})
        package_parent = str(Path(__file__).resolve().parent.parent)
        plan["environment"]["PYTHONPATH"] = os.pathsep.join(filter(None, (package_parent, os.environ.get("PYTHONPATH", ""))))
        plan["environment"]["STEALTH_M2_PROBE_FINAL_SCALE"] = "1"
        plan["environment"]["PYTHONUNBUFFERED"] = "1"
        metadata.update({"status": "running", "plan": plan,
                         "gpu": torch.cuda.get_device_name(0), "torch_version": torch.__version__})
        _json(metadata_path, metadata, replace=True)
        _stream_command(plan, work_dir / "run.log")
        verify_inputs(work_dir)
        artifacts = []
        for index, row in enumerate(record for record in selection["records"] if record["label"] == 1):
            if Path(plan["input_records"][index]["source"]).resolve() != (work_dir / row["canonical_path"]).resolve():
                raise ValueError("Upstream input ordering differs from the frozen probe input manifest")
            for variant, suffix in VARIANTS.items():
                path = work_dir / "outputs" / f"{index:04d}_{suffix}"
                with Image.open(path) as image:
                    if image.size != (224, 224) or image.mode != "RGB":
                        raise ValueError(f"Unexpected output format: {path}")
                    image.verify()
                artifacts.append({"image_id": row["image_id"], "variant": variant,
                                  "path": path.relative_to(work_dir).as_posix(), "sha256": upstream.sha256_file(path)})
            sidecar = work_dir / "outputs" / f"{index:04d}_decoder_probe.json"
            details = json.loads(sidecar.read_text())
            _verify_sidecar(details, work_dir / "outputs", index)
            artifacts.append({"image_id": row["image_id"], "variant": "probe_sidecar",
                              "path": sidecar.relative_to(work_dir).as_posix(), "sha256": upstream.sha256_file(sidecar),
                              "details": details})
        metadata.update({"status": "complete", "finished_at": _now(), "artifacts": artifacts})
    except BaseException as exc:
        metadata.update({"status": "interrupted" if isinstance(exc, KeyboardInterrupt) else "failed",
                         "finished_at": _now(), "error_type": type(exc).__name__, "error": str(exc)})
        if isinstance(exc, subprocess.CalledProcessError):
            metadata["returncode"] = exc.returncode
        _json(metadata_path, metadata, replace=True)
        raise
    _json(metadata_path, metadata, replace=True)
    return metadata


def _verify_sidecar(details: dict, output_dir: Path, index: int) -> None:
    if (details.get("status") != "complete" or details.get("method") != "decoder_probe"
            or details.get("observation_only") is not True or details.get("extra_classifier_calls") != 0):
        raise ValueError("Incomplete or unexpected decoder observation sidecar")
    latent_sha = details.get("latent_sha256")
    if not isinstance(latent_sha, str) or len(latent_sha) != 64:
        raise ValueError("Probe sidecar has no final-latent SHA256")
    observations = dict(details.get("candidates", {}))
    observations["control_vae"] = details.get("final_output", {})
    if observations["control_vae"].get("decoder") != "ControlVAE":
        raise ValueError("Original final output did not use ControlVAE")
    for variant, suffix in VARIANTS.items():
        observation = observations.get(variant, {})
        expected_scale = 0.18215 if variant == "plain_vae" else 0.1825
        if observation.get("latent_sha256") != latent_sha or observation.get("decode_scale") != expected_scale:
            raise ValueError("Decoder observations do not share the declared final latent/scales")
        if observation.get("image_sha256") != upstream.sha256_file(output_dir / f"{index:04d}_{suffix}"):
            raise ValueError("Probe sidecar PNG hash does not match the saved output")


def evaluate_predictions(records, clean_logits, variant_logits, *, detector, quality=None,
                         real_records=(), real_logits=None):
    """Pure metric accounting shared by the saved-image evaluator and CPU tests."""
    import torch
    from .evaluation import aggregate_records, make_prediction_records

    clean = torch.as_tensor(clean_logits).detach().cpu()
    logits = {name: torch.as_tensor(values).detach().cpu() for name, values in variant_logits.items()}
    if set(logits) != set(VARIANTS):
        raise ValueError("All three decoder variants are required")
    rows = []
    for variant in VARIANTS:
        rows.extend(make_prediction_records(records, method=variant, detector=detector, condition="saved_png",
                    raw_clean_logits=clean, clean_logits=clean, attacked_logits=logits[variant],
                    quality=None if quality is None else quality[variant]))
        if real_records:
            if real_logits is None:
                raise ValueError("Real-image logits are required when real records are provided")
            rows.extend(make_prediction_records(real_records, method=variant, detector=detector, condition="saved_png",
                        raw_clean_logits=real_logits, clean_logits=real_logits))
    eligible = clean >= 0
    control_fake = logits["control_vae"] >= 0
    image_indices = {record["image_id"]: index for index, record in enumerate(records)}
    for row in rows:
        row["paired_control_fake"] = None
        row["decoder_transition"] = None
        if row["label"] != 1:
            continue
        index = image_indices[row["image_id"]]
        row["paired_control_fake"] = bool(control_fake[index])
        if row["method"] == "control_vae":
            continue
        if not eligible[index]:
            transition = "clean_already_escaped"
        else:
            plain_success = bool(logits[row["method"]][index] < 0)
            control_success = bool(~control_fake[index])
            transition = ("both_success" if control_success else "success_to_failure") if plain_success else (
                "failure_to_success" if control_success else "both_failure")
        row["decoder_transition"] = transition
    pairs = []
    for variant in ("plain_vae", "plain_vae_final_scale"):
        plain_fake = logits[variant] >= 0
        plain_escape = eligible & ~plain_fake
        control_escape = eligible & ~control_fake
        reversion = int((plain_escape & control_fake).sum())
        reverse = int((eligible & plain_fake & ~control_fake).sum())
        denominator = int(plain_escape.sum())
        eligible_count = int(eligible.sum())
        pairs.append({"detector": detector, "plain_variant": variant,
                      "same_decode_scale_as_control": variant == "plain_vae_final_scale",
                      "eligible_clean_fake_count": eligible_count,
                      "plain_eligible_escape_count": denominator,
                      "control_eligible_escape_count": int(control_escape.sum()),
                      "reversion_count_plain_real_to_control_fake": reversion,
                      "reverse_count_plain_fake_to_control_real": reverse,
                      "both_success_count": int((plain_escape & ~control_fake).sum()),
                      "both_failure_count": int((eligible & plain_fake & control_fake).sum()),
                      "control_minus_plain_success_count": reverse - reversion,
                      "control_minus_plain_conditional_asr": (reverse - reversion) / eligible_count if eligible_count else None,
                      "reversion_rate_given_plain_escape": reversion / denominator if denominator else None})
    return rows, aggregate_records(rows), pairs


def _csv(path: Path, rows: list[dict]) -> None:
    fields = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("x", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def evaluate_probe(work_dir, *, detectors=("E", "R", "D", "S"), device="cuda:0", batch_size=10,
                   detector_loader=None) -> dict:
    """Re-read every saved PNG and evaluate the three final-latent observations."""
    import torch
    from .data import load_rgb
    from .evaluation import image_quality

    work_dir = Path(work_dir).resolve()
    output = work_dir / "evaluation"
    if output.exists():
        raise FileExistsError(f"Evaluation output already exists: {output}")
    detectors = tuple(detectors)
    if not detectors or len(set(detectors)) != len(detectors) or set(detectors) - set(upstream.DETECTOR_WEIGHTS):
        raise ValueError("Choose distinct detector names from R,E,D,S")
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    metadata = json.loads((work_dir / "run_metadata.json").read_text())
    if metadata.get("status") != "complete" or metadata.get("method") != "decoder_probe":
        raise ValueError("Only a successfully completed decoder probe can be evaluated")
    selection = verify_inputs(work_dir)
    if upstream.sha256_file(work_dir / "input_manifest.json") != metadata["input_manifest_sha256"]:
        raise ValueError("Input manifest changed since the probe run")
    preparation = _verify_probe_runtime(work_dir)
    runtime = _runtime_path(work_dir)
    if preparation != metadata["preparation"]:
        raise ValueError("Runtime/checkpoint preparation differs from the completed run")
    generated = [record for record in selection["records"] if record["label"] == 1]
    expected_artifacts = {(row["image_id"], variant) for row in generated for variant in (*VARIANTS, "probe_sidecar")}
    actual_artifacts = {(artifact["image_id"], artifact["variant"]) for artifact in metadata["artifacts"]}
    if actual_artifacts != expected_artifacts or len(actual_artifacts) != len(metadata["artifacts"]):
        raise ValueError("Completed run metadata is missing or duplicates decoder artifacts")
    for artifact in metadata["artifacts"]:
        if upstream.sha256_file(work_dir / artifact["path"]) != artifact["sha256"]:
            raise ValueError(f"Saved probe artifact changed: {artifact['path']}")
    for index in range(len(generated)):
        sidecar = work_dir / "outputs" / f"{index:04d}_decoder_probe.json"
        _verify_sidecar(json.loads(sidecar.read_text()), work_dir / "outputs", index)
    real = [record for record in selection["records"] if record["label"] == 0]
    records = [{"image_id": row["image_id"], "label": 1, "split": "probe", "source": row["source_path"]} for row in generated]
    real_records = [{"image_id": row["image_id"], "label": 0, "split": "probe", "source": row["source_path"]} for row in real]
    clean = torch.stack([load_rgb(work_dir / row["canonical_path"]) for row in generated])
    real_images = torch.stack([load_rgb(work_dir / row["canonical_path"]) for row in real]) if real else None
    variants = {variant: torch.stack([load_rgb(work_dir / "outputs" / f"{index:04d}_{suffix}") for index in range(len(generated))])
                for variant, suffix in VARIANTS.items()}
    quality = {variant: image_quality(clean, images) for variant, images in variants.items()}
    loader = detector_loader or upstream.load_official_detector
    rows, summaries, pairing, detector_hashes = [], [], [], {}
    for name in detectors:
        model = loader(runtime, runtime / "checkpoints", name, device=device,
                       input_size=224, raw_positive_class="real")
        detector_hashes[name] = preparation["checkpoints"][upstream.DETECTOR_WEIGHTS[name]]["sha256"]
        def predict(images):
            with torch.inference_mode():
                return torch.cat([model(batch.to(device)).detach().cpu() for batch in images.split(batch_size)])
        clean_logits = predict(clean)
        variant_logits = {variant: predict(images) for variant, images in variants.items()}
        detector_rows, detector_summaries, detector_pairs = evaluate_predictions(
            records, clean_logits, variant_logits, detector=name, quality=quality,
            real_records=real_records, real_logits=predict(real_images) if real_images is not None else None)
        rows.extend(detector_rows)
        summaries.extend(detector_summaries)
        pairing.extend(detector_pairs)
        del model
        if str(device).startswith("cuda"):
            torch.cuda.empty_cache()
    summary = {"status": "complete", "method": "decoder_probe", "observation_only": True,
               "unchanged_official_baseline": False, "n_generated": len(generated), "n_real": len(real),
               "surrogate": metadata["surrogate"], "detectors": list(detectors),
               "detector_checkpoint_sha256": detector_hashes,
               "run_metadata_sha256": upstream.sha256_file(work_dir / "run_metadata.json"),
               "fake_decision": "fake_logit >= 0 (negated official real-positive head)",
               "metrics": summaries, "decoder_pairing": pairing,
               "interpretation": ["Small exploratory diagnostic, not a new attack method or a final benchmark.",
                  "Conditional ASR uses only these same saved clean images classified as generated by this detector.",
                  "Compare plain_vae_final_scale with control_vae to control the 0.18215 versus 0.1825 scale difference.",
                  "All quality scores compare reloaded saved PNGs with the same frozen original.",
                  "Real-image FPR is unavailable when no real input folder was supplied."]}
    output.mkdir(exist_ok=False)
    try:
        _csv(output / "per_image.csv", rows)
        _csv(output / "summary.csv", summaries)
        _csv(output / "decoder_pairing.csv", pairing)
        _json(output / "summary.json", summary)
    except BaseException:
        shutil.rmtree(output)
        raise
    return summary


def main(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    select = commands.add_parser("select")
    select.add_argument("--generated-root", required=True)
    select.add_argument("--real-root")
    select.add_argument("--work-dir", required=True)
    select.add_argument("--n-images", type=int, default=10)
    select.add_argument("--seed", type=int, default=42)
    prepare = commands.add_parser("prepare")
    prepare.add_argument("--source", required=True)
    prepare.add_argument("--checkpoint-dir", required=True)
    prepare.add_argument("--work-dir", required=True)
    prepare.add_argument("--runtime-dir", help="Local filesystem runtime (recommended for /content); checkpoints may remain on Drive")
    run = commands.add_parser("run")
    run.add_argument("--work-dir", required=True)
    run.add_argument("--surrogate", choices=tuple(upstream.DETECTOR_WEIGHTS), default="E")
    run.add_argument("--iterations", type=int, default=10)
    run.add_argument("--sd-model-id")
    run.add_argument("--python", default=sys.executable)
    evaluate = commands.add_parser("evaluate")
    evaluate.add_argument("--work-dir", required=True)
    evaluate.add_argument("--detectors", default="E,R,D,S")
    evaluate.add_argument("--device", default="cuda:0")
    evaluate.add_argument("--batch-size", type=int, default=10)
    args = parser.parse_args(argv)
    if args.command == "select":
        result = select_inputs(args.generated_root, args.work_dir, n_images=args.n_images, seed=args.seed, real_root=args.real_root)
        brief = {"status": "inputs_selected", "n_generated": result["n_generated"], "n_real": result["n_real"],
                 "input_manifest": str(Path(args.work_dir).resolve() / "input_manifest.json")}
    elif args.command == "prepare":
        result = prepare_probe(args.source, args.checkpoint_dir, args.work_dir, runtime_dir=args.runtime_dir)
        brief = {"status": "runtime_prepared", "method": result["method"], "observation_only": True,
                 "runtime_dir": str(_runtime_path(Path(args.work_dir).resolve())),
                 "runtime_reference": str(Path(args.work_dir).resolve() / "runtime_reference.json")}
    elif args.command == "run":
        result = run_probe(args.work_dir, surrogate=args.surrogate, iterations=args.iterations, sd_model_id=args.sd_model_id, python=args.python)
        brief = {"status": result["status"], "n_generated": result["input_manifest"]["n_generated"],
                 "surrogate": result["surrogate"], "outputs": str(Path(args.work_dir).resolve() / "outputs"),
                 "run_metadata": str(Path(args.work_dir).resolve() / "run_metadata.json"),
                 "run_log": str(Path(args.work_dir).resolve() / "run.log")}
    else:
        result = evaluate_probe(args.work_dir, detectors=tuple(args.detectors.split(",")), device=args.device, batch_size=args.batch_size)
        brief = {"status": result["status"], "n_generated": result["n_generated"], "n_real": result["n_real"],
                 "detectors": result["detectors"], "summary": str(Path(args.work_dir).resolve() / "evaluation/summary.json"),
                 "per_image": str(Path(args.work_dir).resolve() / "evaluation/per_image.csv")}
    print(json.dumps(brief, indent=2, ensure_ascii=False, allow_nan=False))


if __name__ == "__main__":
    main()


## 4. 建立与原 Notebook 一致的独立环境
核心版本沿用原第 3.3 节：Python 3.10、torch 2.2.2 / CUDA 12.1、diffusers 0.12.1。
安装和下载属于首次准备成本；不当作逐图攻击耗时。失败会停止并显示错误。


In [ ]:
import shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
PYTHON = '/content/sdenv/bin/python'
if not Path(PYTHON).exists():
    subprocess.run([UV, 'venv', '--python', '3.10', '/content/sdenv'], check=True)
subprocess.run([UV, 'pip', 'install', '--python', PYTHON,
                'torch==2.2.2', 'torchvision==0.17.2',
                '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([UV, 'pip', 'install', '--python', PYTHON,
                'numpy==1.26.4', 'diffusers==0.12.1', 'huggingface_hub==0.23.4',
                'transformers==4.30.2', 'accelerate==0.20.3', 'timm==0.6.12',
                'efficientnet-pytorch==0.7.1', 'torchattacks==3.5.1', 'lossers==0.0.9',
                'eagerpy==0.30.0', 'natsort==8.4.0', 'tensorboardX==2.6.2.2',
                'scipy==1.11.4', 'scikit-learn==1.3.2',
                'opencv-python-headless==4.10.0.84', 'Pillow==10.4.0', 'tqdm==4.66.1'], check=True)
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(PROJECT / 'm2')
ENV['PYTHONHASHSEED'] = str(SEED)
ENV['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
def execute(arguments):
    subprocess.run([PYTHON, *map(str, arguments)], cwd=PROJECT, env=ENV, check=True)
execute(['-c', "import torch, diffusers, timm; from lossers.lpips import LPIPS; assert torch.cuda.is_available(), '请选择 GPU'; print('GPU:',torch.cuda.get_device_name(0),'显存 GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,1)); print('torch',torch.__version__,'diffusers',diffusers.__version__)"])


## 5. 固定官方代码版本
原第 3.1 / 3.5 节的下载与单 GPU 兼容处理在这里执行；额外的观察插桩只允许匹配已核查源码。


In [ ]:
UPSTREAM_COMMIT = 'a156fb0792732e1aabc5b94a0cbecf6829e79d73'
REFERENCE = PROJECT / 'third_party/reference'
if not REFERENCE.exists():
    subprocess.run(['git', 'init', str(REFERENCE)], check=True)
    subprocess.run(['git', '-C', str(REFERENCE), 'remote', 'add', 'origin', 'https://github.com/wyczzy/StealthDiffusion.git'], check=True)
    subprocess.run(['git', '-C', str(REFERENCE), 'fetch', '--depth', '1', 'origin', UPSTREAM_COMMIT], check=True)
    subprocess.run(['git', '-C', str(REFERENCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
assert subprocess.check_output(['git', '-C', str(REFERENCE), 'rev-parse', 'HEAD'], text=True).strip() == UPSTREAM_COMMIT


## 6. 选择输入并连接权重
保留源图路径与哈希、固定选择清单和实际 224×224 输入。只读源图，不补造图片。
准备步骤仍检查原 Notebook 的六项资产；文件缺失会给出名称。


In [ ]:
args = ['-m', 'stealth_m2.decoder_probe_run', 'select', '--generated-root', GENERATED_ROOT,
        '--work-dir', RUN_DIR, '--n-images', N_IMAGES, '--seed', SEED]
if REAL_ROOT.strip():
    args += ['--real-root', REAL_ROOT]
execute(args)
execute(['-m', 'stealth_m2.decoder_probe_run', 'prepare', '--source', REFERENCE,
         '--checkpoint-dir', CKPT_SRC, '--work-dir', RUN_DIR,
         '--runtime-dir', PROJECT / 'runtimes' / RUN_DIR.name])
(RUN_DIR / 'notebook_provenance.json').write_text(json.dumps({
    'source_manifest_sha256': SOURCE_MANIFEST_SHA256,
    'parent_notebook': 'StealthDiffusion_Baseline_Submission_1.ipynb',
    'parent_notebook_sha256': '84441e2d331e44ac03873f524d56375f4d309ded4261da8d2bfb4e7732dfe1e6',
    'upstream_commit': UPSTREAM_COMMIT, 'purpose': 'exploratory_decoder_path_diagnostic',
    'new_attack_implemented': False}, indent=2))


## 7. 运行这 10 张图的官方攻击与观察
默认仍用 10 次 LAO、DDIM20、最后 2 步、224 输入，与原 Notebook 第 3.6 节一致。
这里只运行一个代理攻击，每张图额外保存两张普通 VAE 候选。最终 Control-VAE 路径原样保留。
日志完整保存在输出目录；报错不视为完成。观察额外解码的时间单独记录。


In [ ]:
execute(['-m', 'stealth_m2.decoder_probe_run', 'run', '--work-dir', RUN_DIR,
         '--surrogate', SURROGATE, '--iterations', ITERATIONS,
         '--sd-model-id', SD_MODEL_ID, '--python', PYTHON])


## 8. 重新读取 PNG 进行配对评测
所有阶段都使用相同检测器预处理与固定阈值。条件 ASR 的分母只包含原图被正确识别为 AI 的图像；
原图已逃逸的案例不计为攻击功劳。真实照片未提供时 FPR 为不可计算。
本阶段查看 R/E/D/S，因此后续不能把这些结果称为完全未接触的最终测试。


In [ ]:
execute(['-m', 'stealth_m2.decoder_probe_run', 'evaluate', '--work-dir', RUN_DIR,
         '--detectors', 'E,R,D,S', '--device', 'cuda:0', '--batch-size', 10])


## 9. 查看结果和图片
`per_image.csv` 保留每张图的分数、成功变化与画质；`summary.csv` 汇总。
比较普通路径与最终路径时，同时看回退、恢复和画质；仅回退计数不能说明模块总体有害。


In [ ]:
import csv
from IPython.display import display, Markdown
summary = json.loads((RUN_DIR / 'evaluation/summary.json').read_text())
labels = {'plain_vae': '普通 VAE', 'plain_vae_final_scale': '普通 VAE，同缩放', 'control_vae': '最终 Control-VAE'}
def number(value, digits=3):
    return 'n/a' if value is None else f'{value:.{digits}f}'
def percentage(value):
    return 'n/a' if value is None else f'{100*value:.1f}%'
table = ['| 检测器 | 输出 | 成功 / 原图检测正确 | 条件 ASR | PSNR | SSIM | Linf |',
         '|---|---|---|---|---|---|---|']
for row in summary['metrics']:
    table.append('| ' + ' | '.join([row['detector'], labels[row['method']],
        f"{row['success_count']}/{row['eligible_count']}", percentage(row['conditional_asr']),
        number(row['mean_psnr'],2), number(row['mean_ssim']), number(row['mean_linf'],4)]) + ' |')
display(Markdown('\n'.join(table)))
pairs = ['| 检测器 | 普通候选 | 原图检测正确 | 成功→失败 | 失败→成功 | 净增加成功数 |',
         '|---|---|---|---|---|---|']
for row in summary['decoder_pairing']:
    back = row['reversion_count_plain_real_to_control_fake']
    gain = row['reverse_count_plain_fake_to_control_real']
    pairs.append('| ' + ' | '.join(map(str, [row['detector'], labels[row['plain_variant']],
        row['eligible_clean_fake_count'], back, gain, gain-back])) + ' |')
display(Markdown('\n'.join(pairs)))
for row in summary['metrics']:
    if row['method'] == 'control_vae':
        print(row['detector'], '原图AI识别:', f"{row['clean_fake_detected_count']}/{row['n_generated']}",
              '| 真实照片误报率:', percentage(row['real_fpr']))
print('完整汇总:', RUN_DIR / 'evaluation/summary.json')
print('逐图记录:', RUN_DIR / 'evaluation/per_image.csv')
from PIL import Image
with (RUN_DIR / 'evaluation/per_image.csv').open(newline='') as handle:
    per_image = list(csv.DictReader(handle))
reverted = [row for row in per_image if row.get('detector') == SURROGATE
            and row.get('method') == 'plain_vae' and row.get('decoder_transition') == 'success_to_failure']
example_id = reverted[0]['image_id'] if reverted else '0000'
stage_files = [RUN_DIR / 'outputs' / (example_id + '_' + name) for name in
               ('originImage.png', 'plain_vae.png', 'plain_vae_final_scale.png', 'adv_image.png')]
canvas = Image.new('RGB', (224*4,224), 'white')
for index, path in enumerate(stage_files):
    with Image.open(path) as im:
        canvas.paste(im.convert('RGB').resize((224,224)), (224*index,0))
display(Markdown('示例 ' + example_id + '：**原图 → 普通 VAE → 普通 VAE（最终缩放）→ 最终 Control-VAE**'))
display(canvas)


## 10. 如何决定下一步
- 有“原图检测正确、普通候选逃逸、最终图重新被识别”的配对案例：支持下一步尝试只修改 LAO 的解码路径损失，再与原版比较。
- 最终路径同时恢复其他样本的成功：报告两种变化和整体净变化，不只展示失败案例。
- 没有回退：仅表示本批次没有观察到这个现象，暂不将它当作改进动机。
- 原始图大多已逃逸，或两种输出均失败：当前样本对该问题提供的信息有限，先核对输入来源和模型适用性。

本次只是小样本探索，不构成统计显著的性能提升、最终 M2 基准或已完成的对方队伍互动。
输出 PSNR/SSIM/Linf 是最终图像的测量值；初始 PGD 的 4/255 不约束整个扩散输出。
先查看这个实验，再决定是否推进单项 A；不在本次追加多代理、EOT 或重新训练 Control-VAE。


## 参考
- 原上传 Notebook：`StealthDiffusion_Baseline_Submission_1.ipynb`；项目中的参考副本保持原字节。
- [StealthDiffusion 原论文](https://arxiv.org/abs/2408.05669)
- [固定的官方攻击实现](https://github.com/wyczzy/StealthDiffusion/blob/a156fb0792732e1aabc5b94a0cbecf6829e79d73/diff_latent_attack.py)
